In [ ]:
import os
import glob
import math
import re
import random
import logging
import warnings
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image
import torchvision.transforms as transforms
from torchvision.models.segmentation import deeplabv3_resnet101
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import cv2

from peft import LoraConfig, get_peft_model, PeftModel
from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    get_linear_schedule_with_warmup,
)
from torch.cuda.amp import GradScaler, autocast


warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
torch.backends.cuda.matmul.allow_tf32 = True



def get_segmentation_model() -> nn.Module:

    model = deeplabv3_resnet101(weights='DeepLabV3_ResNet101_Weights.DEFAULT')
    model.classifier[4] = nn.Conv2d(256, 1, kernel_size=(1, 1), stride=(1, 1))
    return model

def get_segmentation_transforms() -> A.Compose:

    return A.Compose([
        A.Resize(256, 256),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def post_process_mask(mask: np.ndarray, kernel_size: int = 5, min_area: int = 100) -> np.ndarray:

    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    opened_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    closed_mask = cv2.morphologyEx(opened_mask, cv2.MORPH_CLOSE, kernel)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(closed_mask, connectivity=8)
    processed_mask = np.zeros(mask.shape, dtype=np.uint8)
    if num_labels > 1:
        largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        if stats[largest_label, cv2.CC_STAT_AREA] > min_area:
            processed_mask[labels == largest_label] = 255

    return processed_mask.astype(np.uint8)

def delineate_roi_on_image(pil_image: Image.Image, seg_model: nn.Module, seg_transform: A.Compose, device: str) -> Tuple[Image.Image, bool]:

    open_cv_image = np.array(pil_image.convert("RGB"))

    augmented = seg_transform(image=open_cv_image)
    image_tensor = augmented['image'].to(device).unsqueeze(0)

    seg_model.eval()
    with torch.no_grad():
        output = seg_model(image_tensor)['out']

    mask = torch.sigmoid(output).squeeze().cpu().numpy()
    binary_mask = (mask > 0.5).astype(np.uint8)
    cleaned_mask = post_process_mask(binary_mask)

    contours, _ = cv2.findContours(cleaned_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    has_tumor = bool(contours)
    if has_tumor:
        cv2.drawContours(open_cv_image, contours, -1, (0, 255, 255), 2)

    return Image.fromarray(open_cv_image), has_tumor


class VLM_QADataset(Dataset):
    def __init__(self, image_paths: List[str], metadata_df: pd.DataFrame, seg_model: nn.Module, seg_transform: A.Compose, device: str, is_train: bool = True):
        self.image_paths_map: Dict[str, str] = {}
        self.metadata_df = metadata_df.set_index("Patient")
        self.seg_model = seg_model
        self.seg_transform = seg_transform
        self.device = device
        self.is_train = is_train

        self.vlm_transform = transforms.Compose([transforms.Resize((336, 336))])


        for img_path in image_paths:
            pid_folder = os.path.basename(os.path.dirname(img_path))
            pid_key = "_".join(pid_folder.split("_")[0:3])
            if pid_key in self.metadata_df.index:
                row = self.metadata_df.loc[[pid_key]].iloc[0]
                grade = row.get("neoplasm_histologic_grade")
                if pd.notna(grade) and int(grade) in [1, 2]:
                    self.image_paths_map[img_path] = pid_key

        self.valid_paths = list(self.image_paths_map.keys())

    def __len__(self) -> int:
        return len(self.valid_paths)

    def __getitem__(self, idx: int):
        img_path = self.valid_paths[idx]
        image_pil = Image.open(img_path).convert("RGB")


        delineated_image, seg_found_tumor = delineate_roi_on_image(image_pil, self.seg_model, self.seg_transform, self.device)
        final_image = self.vlm_transform(delineated_image)

        pid_key = self.image_paths_map[img_path]
        row = self.metadata_df.loc[[pid_key]].iloc[0]
        grade = row.get("neoplasm_histologic_grade")

        q, a = "", ""


        if self.is_train:

            if seg_found_tumor:
                q = "What is the histologic grade of the brain tumor (delineated) in the MRI: one or two?"
                a = f"The grade of the tumor is {'two' if int(grade) == 2 else 'one'}."
            else:
                q = "Is a tumor visible in the delineated region of the MRI?"
                a = "No tumor is visible."
        else:

            mask_path = img_path.replace('.tif', '_mask.tif')
            gt_mask = np.array(Image.open(mask_path).convert("L"))
            gt_has_tumor = np.any(gt_mask > 0)

            if gt_has_tumor:
                q = "What is the histologic grade of the brain tumor (delineated) in the MRI: one or two?"
                a = f"The grade of the tumor is {'two' if int(grade) == 2 else 'one'}."
            else:
                q = "Is a tumor visible in the delineated region of the MRI?"
                a = "No tumor is visible."

        return final_image, q, a



def vlm_collate_fn_for_training(batch):
    images, questions, answers = zip(*batch)
    return list(images), list(questions), list(answers)

def vlm_collate_fn_for_evaluation(batch):
    images, questions, answers = zip(*batch)
    return list(images), list(questions), list(answers)

def build_training_batch_cpu_main(images, questions, answers, processor: AutoProcessor):
    prompts = [f"USER: <image>\n{q}\nASSISTANT:" for q in questions]
    full_texts = [f"USER: <image>\n{q}\nASSISTANT: {a}{processor.tokenizer.eos_token}" for q, a in zip(questions, answers)]
    toks_prompt = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    toks_full = processor(text=full_texts, images=images, return_tensors="pt", padding=True)
    labels = toks_full.input_ids.clone()
    prompt_lens = torch.sum(toks_prompt.attention_mask, dim=1)
    for i in range(labels.size(0)):
        labels[i, : prompt_lens[i]] = -100
    labels[labels == processor.tokenizer.pad_token_id] = -100
    return {
        "input_ids": toks_full.input_ids, "pixel_values": toks_full.pixel_values,
        "attention_mask": toks_full.attention_mask, "labels": labels,
    }



def _assistant_span(text: str) -> str:
    if not isinstance(text, str): return ""
    parts = text.split("ASSISTANT:")
    return (parts[-1] if parts else text).strip().lower()

def _has_one_two_flags(answer_text: str) -> Tuple[bool, bool]:
    answer_text = answer_text.replace("\u2019", "'")
    tokens = set(re.findall(r"\b(one|two|1|2)\b", answer_text))
    return ("one" in tokens) or ("1" in tokens), ("two" in tokens) or ("2" in tokens)

def compute_token_accuracy_shifted(logits: torch.Tensor, labels: torch.Tensor, eos_id: int = None) -> tuple:
    with torch.no_grad():
        logits = logits[:, :-1, :]; labels = labels[:, 1:]
        if eos_id is not None:
            labels = labels.clone(); labels[labels == eos_id] = -100
        preds = torch.argmax(logits, dim=-1); mask = labels != -100
        return (preds[mask] == labels[mask]).sum().item(), mask.sum().item()

def run_evaluation(model, processor, data_loader: DataLoader, device, description="Evaluating"):
    model.eval()
    vlm_correct, total_samples = 0, 0
    total_loss_sum, total_loss_count = 0.0, 0
    total_tok_correct, total_tok_count = 0, 0
    debug_printed = False

    with torch.no_grad():
        for batch in tqdm(data_loader, desc=description):
            images, questions, answers = batch
            prompts = [f"USER: <image>\n{q}\nASSISTANT:" for q in questions]
            with autocast():
                gen_inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True).to(device)
                generated_ids = model.generate(**gen_inputs, max_new_tokens=20, pad_token_id=processor.tokenizer.pad_token_id)
            decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)

            for i in range(len(decoded)):
                pred_span = _assistant_span(decoded[i])
                true_answer = answers[i]
                is_no_tumor_case = "no tumor" in true_answer.lower()
                ok = False
                if is_no_tumor_case:
                    if "no tumor" in pred_span: ok = True
                else:
                    want_two = "two" in true_answer
                    has_one, has_two = _has_one_two_flags(pred_span)
                    if (want_two and has_two and not has_one) or ((not want_two) and has_one and not has_two):
                        ok = True

                if not debug_printed:
                    print(f"\n[DEBUG]\n  pred_raw=\n{decoded[i]}\n  pred_span=\n{pred_span}\n  true=\n{answers[i]}\n  is_no_tumor_case={is_no_tumor_case} -> ok={ok}")
                if ok: vlm_correct += 1

            full_texts = [f"USER: <image>\n{q}\nASSISTANT: {a}{processor.tokenizer.eos_token}" for q, a in zip(questions, answers)]
            toks_prompt = processor(text=prompts, images=images, return_tensors="pt", padding=True)
            toks_full = processor(text=full_texts, images=images, return_tensors="pt", padding=True)
            labels = toks_full.input_ids.clone()
            prompt_lens = torch.sum(toks_prompt.attention_mask, dim=1)
            for i in range(labels.size(0)): labels[i, : prompt_lens[i]] = -100
            labels[labels == processor.tokenizer.pad_token_id] = -100

            with autocast():
                ce_inputs = {
                    "input_ids": toks_full.input_ids.to(device),
                    "pixel_values": toks_full.pixel_values.to(device, dtype=torch.float16),
                    "attention_mask": toks_full.attention_mask.to(device), "labels": labels.to(device),
                }
                out = model(**ce_inputs, return_dict=True)
                total_loss_sum += out.loss.item()
                total_loss_count += 1
                c, n = compute_token_accuracy_shifted(out.logits.detach(), labels.to(out.logits.device), eos_id=processor.tokenizer.eos_token_id)
                total_tok_correct += c; total_tok_count += n

            total_samples += len(answers)
            debug_printed = True

    vlm_acc = (vlm_correct / total_samples) * 100 if total_samples else 0.0
    avg_loss = (total_loss_sum / total_loss_count) if total_loss_count else float("inf")
    ppl = math.exp(avg_loss) if avg_loss < 50 else float("inf")
    tok_acc = (total_tok_correct / total_tok_count) * 100 if total_tok_count else 0.0

    print(f"\n--- Results for {description} ---")
    print(f"  - VLM Accuracy (QA):          {vlm_acc:.2f}%")
    print(f"  - Perplexity (teacher-forced):  {ppl:.4f}")
    print(f"  - Token Accuracy (answer-only): {tok_acc:.2f}%")
    print("-" * 40)
    return vlm_acc, ppl, tok_acc



def discover_lora_targets(llava_model, include_vision: bool = True) -> List[str]:
    text_keys = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
    projector_keys = {"multi_modal_projector"}
    vision_keys = {"q_proj", "k_proj", "v_proj", "out_proj"}
    target_suffixes: set[str] = set()
    for name, module in llava_model.named_modules():
        if any(k in name for k in text_keys): target_suffixes.add(name.split(".")[-1])
        if any(k in name for k in projector_keys):
            if hasattr(module, "weight") and getattr(module, "weight", None) is not None:
                target_suffixes.add(name.split(".")[-1])
        if include_vision and ("vision_tower" in name) and any(k in name for k in vision_keys):
            target_suffixes.add(name.split(".")[-1])
    return sorted(target_suffixes if target_suffixes else text_keys)


if __name__ == "__main__":
    config = {
        "device": "cuda:2" if torch.cuda.is_available() else "cpu",
        "base_path": "/home/ealam/Downloads/LGG dataset Cameron/lgg-mri-segmentation/kaggle_3m",
        "local_llava_path": "/home/ealam/Desktop/llava-1.5-7b-local",
        "save_path": "./llava-lora-delineated-conditional-qa",
        "csv_path": "/home/ealam/Downloads/LGG dataset Cameron/lgg-mri-segmentation/kaggle_3m/data.csv",
        "segmentation_model_path": "best_model_segmentation_v2.pth",
        "learning_rate": 1e-4, "batch_size": 4, "num_epochs": 25,
        "early_stopping_patience": 5, "seed": 42,
        "include_vision_lora": True, "num_workers": 0,
    }

    torch.manual_seed(config["seed"]); np.random.seed(config["seed"]); random.seed(config["seed"])
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(config["seed"])
    DEVICE = config["device"]

    print("Step 1: Loading the pre-trained segmentation model...")
    if not os.path.exists(config['segmentation_model_path']):
        raise FileNotFoundError(f"Segmentation model not found at: {config['segmentation_model_path']}.")
    seg_model = get_segmentation_model()
    seg_model.load_state_dict(torch.load(config['segmentation_model_path'], map_location=DEVICE))
    seg_model.to(DEVICE).eval()
    seg_transform = get_segmentation_transforms()
    print("Segmentation model loaded successfully.")

    print("\nStep 2: Gathering and splitting data...")


    base_path = config["base_path"]
    if not os.path.exists(base_path):
        raise FileNotFoundError(
            f"The specified base_path does not exist: {base_path}\n"

        )
    print(f"Checking for images in: {base_path}")

    all_image_paths = [p.replace("_mask.tif", ".tif") for p in glob.glob(os.path.join(config["base_path"], "**", "*_mask.tif"), recursive=True)]
    all_image_paths = [p for p in all_image_paths if os.path.exists(p)]


    if not all_image_paths:
        print("\n--- ERROR: No images found ---")
        exit()

    print(f"Found {len(all_image_paths)} total images.")
    usable_paths, _ = train_test_split(all_image_paths, test_size=0.01, random_state=config["seed"])
    train_val_paths, test_paths = train_test_split(usable_paths, test_size=0.20, random_state=config["seed"])
    train_paths, val_paths = train_test_split(train_val_paths, test_size=0.20, random_state=config["seed"])
    print(f"Splitting into {len(train_paths)} train, {len(val_paths)} val, and {len(test_paths)} test.")

    print("\nStep 3: Setting up VLM model and processor...")
    base_model = LlavaForConditionalGeneration.from_pretrained(config["local_llava_path"], torch_dtype=torch.float16, low_cpu_mem_usage=True)
    processor = AutoProcessor.from_pretrained(config["local_llava_path"])
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.add_special_tokens({"pad_token": "[PAD]"})
        base_model.resize_token_embeddings(len(processor.tokenizer))

    target_modules = discover_lora_targets(base_model, include_vision=config["include_vision_lora"])
    print("LoRA target modules:", target_modules)
    lora_cfg = LoraConfig(r=32, lora_alpha=64, target_modules=target_modules, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
    peft_model = get_peft_model(base_model, lora_cfg).to(DEVICE)
    for name, p in peft_model.named_parameters():
        if "multi_modal_projector" in name:
            p.requires_grad = True
            if p.dtype != torch.float32: p.data = p.data.to(torch.float32)
    peft_model.print_trainable_parameters()

    print("\nStep 4: Preparing DataLoaders...")
    metadata_df = pd.read_csv(config["csv_path"])
    train_ds = VLM_QADataset(train_paths, metadata_df, seg_model, seg_transform, DEVICE, is_train=True)
    val_ds = VLM_QADataset(val_paths, metadata_df, seg_model, seg_transform, DEVICE, is_train=False)
    test_ds = VLM_QADataset(test_paths, metadata_df, seg_model, seg_transform, DEVICE, is_train=False)

    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, num_workers=config["num_workers"], pin_memory=True, collate_fn=vlm_collate_fn_for_training)
    val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False, num_workers=config["num_workers"], pin_memory=True, collate_fn=vlm_collate_fn_for_evaluation)
    test_loader = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False, num_workers=config["num_workers"], pin_memory=True, collate_fn=vlm_collate_fn_for_evaluation)

    print("\nStep 5: Starting fine-tuning with LoRA...")
    optimizer = AdamW((p for p in peft_model.parameters() if p.requires_grad), lr=config["learning_rate"])
    scaler = GradScaler()
    num_training_steps = len(train_loader) * config["num_epochs"]
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(num_training_steps * 0.1), num_training_steps=num_training_steps)
    best_val_acc, patience = 0.0, 0

    def _to_device(batch_cpu, device):
        out = {}
        for k, v in batch_cpu.items():
            if k == "pixel_values": out[k] = v.to(device, dtype=torch.float16, non_blocking=True)
            elif torch.is_tensor(v): out[k] = v.to(device, non_blocking=True)
            else: out[k] = v
        return out

    for epoch in range(config["num_epochs"]):
        peft_model.train(); total_loss = 0.0
        for images, questions, answers in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
            batch_cpu = build_training_batch_cpu_main(images, questions, answers, processor)
            batch = _to_device(batch_cpu, DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                out = peft_model(**batch, return_dict=True)
                loss = out.loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()

        print(f"\nEpoch {epoch+1} Avg Loss -> {total_loss / max(1, len(train_loader)):.4f}")
        val_acc, _, _ = run_evaluation(peft_model, processor, val_loader, DEVICE, description="Validation Set Eval")

        if val_acc > best_val_acc:
            print(f"  -> New best validation accuracy ({val_acc:.2f}%). Saving adapters...")
            best_val_acc = val_acc; patience = 0
            peft_model.save_pretrained(config["save_path"])
            processor.save_pretrained(config["save_path"])
        else:
            patience += 1
            print(f"  -> No improvement for {patience} epoch(s).")
            if patience >= config["early_stopping_patience"]:
                print("\n--- Early stopping triggered. ---"); break
        print("=" * 80)

    print("\nStep 6: Loading best adapters for final evaluation...")
    if os.path.exists(config["save_path"]):
        base = LlavaForConditionalGeneration.from_pretrained(config["local_llava_path"], torch_dtype=torch.float16, low_cpu_mem_usage=True)
        final_peft = PeftModel.from_pretrained(base, config["save_path"]).to(DEVICE)
        run_evaluation(final_peft, processor, test_loader, DEVICE, description="Final Test Evaluation")
    else:
        print("No adapters were saved.")





Step 1: Loading the pre-trained segmentation model...
Segmentation model loaded successfully.

Step 2: Gathering and splitting data...
Checking for images in: /home/ealam/Downloads/LGG dataset Cameron/lgg-mri-segmentation/kaggle_3m
Found 3929 total images.
Splitting into 2488 train, 623 val, and 778 test.

Step 3: Setting up VLM model and processor...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

LoRA target modules: ['down_proj', 'gate_proj', 'k_proj', 'linear_1', 'linear_2', 'o_proj', 'out_proj', 'q_proj', 'up_proj', 'v_proj']
trainable params: 107,651,072 || all params: 7,150,098,432 || trainable%: 1.5056

Step 4: Preparing DataLoaders...

Step 5: Starting fine-tuning with LoRA...


Training Epoch 1: 100%|███████████████████████| 608/608 [11:11<00:00,  1.10s/it]



Epoch 1 Avg Loss -> 0.2986


...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Final Test Evaluation:   0%|                            | 0/191 [00:00<?, ?it/s]


[DEBUG]
  pred_raw=
USER:  
What is the histologic grade of the brain tumor (delineated) in the MRI: one or two?
ASSISTANT: The grade of the tumor is two.
  pred_span=
the grade of the tumor is two.
  true=
The grade of the tumor is two.
  is_no_tumor_case=False -> ok=True

[DEBUG]
  pred_raw=
USER:  
What is the histologic grade of the brain tumor (delineated) in the MRI: one or two?
ASSISTANT: The grade of the tumor is two.
  pred_span=
the grade of the tumor is two.
  true=
The grade of the tumor is two.
  is_no_tumor_case=False -> ok=True

[DEBUG]
  pred_raw=
USER:  
Is a tumor visible in the delineated region of the MRI?
ASSISTANT: No tumor is visible.
  pred_span=
no tumor is visible.
  true=
No tumor is visible.
  is_no_tumor_case=True -> ok=True

[DEBUG]
  pred_raw=
USER:  
Is a tumor visible in the delineated region of the MRI?
ASSISTANT: No tumor is visible.
  pred_span=
no tumor is visible.
  true=
No tumor is visible.
  is_no_tumor_case=True -> ok=True


Final Test Evaluation: 100%|██████████████████| 191/191 [04:02<00:00,  1.27s/it]


--- Results for Final Test Evaluation ---
  - VLM Accuracy (QA):          87.94%
  - Perplexity (teacher-forced):  1.0173
  - Token Accuracy (answer-only): 99.16%
----------------------------------------
